## Análise Temporal por Semestre

Perguntas:
- Picos de Venda vs. Picos de Sinistro: 
    - Existe algum trimestre específico onde a venda de seguros aumenta, mas os sinistros aumentam desproporcionalmente? (Ex: vende-se muito no T4, mas há muitos acidentes no T1?)
- Tendência do Ticket Médio: 
    - A média do valor_premio ou do capital_segurado está subindo ou descendo ao longo dos trimestres? A regressão linear indica crescimento sustentável?
- Concentração Temporal de Sinistros:
     - Certos tipos de sinistro (tipo_sinistro) ocorrem mais em semestres específicos? (Isso ajuda a prever reservas financeiras para pagamentos).
- Estabilidade da Carteira: 
    - A proporção de seguros ativos (status_seguro) mantém-se constante ao longo do ano ou há trimestres com maior taxa de cancelamento/não renovação?

### Vamos carregar o dataset

In [0]:
import os

current_path = os.getcwd()
repo_name = "Grupo7-Setor-de-Seguros"

if repo_name in current_path:
    root_path = current_path.split(repo_name)[0] + repo_name
else:
    root_path = os.path.dirname(os.path.dirname(os.getcwd()))

caminho_arquivo = f"{root_path}/data/processed/prata/seguros_sinistros.csv"

print(f"Diretório Raiz: {root_path}")
print(f"Arquivo Alvo: {caminho_arquivo}")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, BooleanType, DateType, LongType

schema_seguros = StructType([
    StructField("nome_contratante", StringType(), True),
    StructField("estado_contratante", StringType(), True),
    StructField("data_contratacao", DateType(), True),
    StructField("valor_pagamento", DoubleType(), True),
    StructField("valor_premio", DoubleType(), True),
    StructField("nome_beneficiario", StringType(), True),
    StructField("status_apolice", StringType(), True),
    StructField("Cobertura1", StringType(), True),
    StructField("Valor Cob 1", DoubleType(), True),
    StructField("Cobertura2", StringType(), True),
    StructField("Valor Cob 2", DoubleType(), True),
    StructField("Cobertura3", StringType(), True),
    StructField("Valor Cob 3", DoubleType(), True),
    StructField("capital_segurado", DoubleType(), True),
    StructField("tipo_sinistro", StringType(), True),
    StructField("valor_sinistro", DoubleType(), True),
    StructField("quem_forma_beneficiados", StringType(), True),
    StructField("status_seguro", StringType(), True),
    StructField("regiao_sinistro", StringType(), True),
    StructField("REGIAO", StringType(), True),
    StructField("SEXO", StringType(), True),
    StructField("TRIMESTRE", IntegerType(), True),
    StructField("ACIMA_DE_3_QUARTIL_PREMIO", BooleanType(), True),
    StructField("ABAIXO_DE_1_QUARTIL_PREMIO", BooleanType(), True),
    StructField("ACIMA_DE_3_QUARTIL_CAPITAL", BooleanType(), True),
    StructField("ABAIXO_DE_1_QUARTIL_CAPITAL", BooleanType(), True),
    StructField("QTD_ACIDENTES_POR_NOME_SEGURADO", LongType(), True),
    StructField("QTD_ACIDENTES_POR_NOME_CONTRATANTE", LongType(), True),
    StructField("RAZÃO_PAGAMENTO_PREMIO", DoubleType(), True),
    StructField("RAZÃO_PAGAMENTO_CAPITAL", DoubleType(), True)
])


df_seguros = spark.read \
    .format("csv") \
    .schema(schema_seguros) \
    .option("header", "true") \
    .option("sep", ",") \
    .option("dateFormat", "yyyy-MM-dd") \
    .load(caminho_arquivo)

df_seguros.printSchema()
display(df_seguros)

---
#### Trimestres
- A primeira análise considerará os trimestres por anos (ex. 1 trim. 2020 != 1 trim. 2021)
- Para isso, vamos criar as colunas necessárias

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

df_base = df_seguros.withColumn("teve_sinistro", F.when(F.col("regiao_sinistro") == "null", 0).otherwise(1))

# Trimestre_Unico: String para visualização (ex: 2021-Q1)
df_tempo = df_base \
    .withColumn("ano", F.year("data_contratacao")) \
    .withColumn("trimestre_unico", F.concat_ws("-", F.col("ano"), F.col("TRIMESTRE"))) \

display(df_tempo)

---
#### Seguros e Sinistros

- Objetivo:
    - Calcular quantidade de seguros em cada trimestre.
    - Calcular quantidade de sinistros em cada trimestre.
    - Calcular as quantidades de cada tipo de sinistro em cada trimestre.

In [0]:
df_trimestral = df_tempo.groupBy("ano", "TRIMESTRE", "trimestre_unico").agg(
    F.count("*").alias("qtd_seguros"),
    F.sum("teve_sinistro").alias("qtd_sinistros")
).orderBy("ano", "TRIMESTRE")

display(df_trimestral)

Databricks visualization. Run in Databricks to view.

In [0]:
# --- 2.2 Quantidade por Tipo de Sinistro em cada Trimestre ---
df_tipos_sinistro = df_tempo.filter("teve_sinistro = 1") \
    .groupBy("trimestre_unico", "tipo_sinistro") \
    .count() \
    .orderBy("trimestre_unico", "tipo_sinistro")

display(df_tipos_sinistro)

Databricks visualization. Run in Databricks to view.

---
#### Resultado

1. Percebemos que, exceto o primeiro e o último semestre (provavelmente por falta de dados), o número de seguros e sinitros se manteve constante ao longo do tempo.

2. Em relação aos tipos de sinitros, percebe-se que independentemente do semestre o padrão é 
- um alto número de mortes acidentais, seguido por 
- um número significativamente menor de invalidez permanente e, finalmente, 
- quantidades parecidas de doença grave e auxílio funeral

---
### Representividade do tipo de sinistro e status por trimestre

- Objetivo
  - Para cada trimestre e cada tipo de sinistro, calcular a razão entre o número sinistros no trimestre e o número total durante o ano.
  - Para cada trimestre e cada status de seguro, calcular a razão entre o número de seguros com este status no trimestre e o número total durante o ano.

Buscamos ver se algum trimestre tem maior ou menos representatividade de algum tipo de sinistro ou status em relação às quantidades totais do ano.

In [0]:
# --- 3.1 Razão Sinistros (Trimestre / Ano) por TIPO ---

# Ano por tipo de sinistro
window_ano_tipo = Window.partitionBy("ano", "tipo_sinistro")

df_razao_sinistro = df_tempo.filter("teve_sinistro = 1") \
    .groupBy("ano", "TRIMESTRE", "tipo_sinistro").count().withColumnRenamed("count", "qtd_trimestre") \
    .withColumn("total_ano", F.sum("qtd_trimestre").over(window_ano_tipo)) \
    .withColumn("razao_trimestral", F.col("qtd_trimestre") / F.col("total_ano")) \
    .orderBy("ano", "tipo_sinistro", "trimestre")

display(df_razao_sinistro)

### Resultado

1. Auxílio Funeral
- Em relação aos anos, nota-se um padrão de leve crescimento ao longo dos semestres
- Nota-se uma leve queda de casos totais de auxílio funeral ao longo dos anos: 2782 -> 2671 -> 2300

2. Doença Grave, Invalidez Permanentee Morte Acidental

4. 2025
- O quarto trimestre de 2025 teve poucos sinistros no geral, o que pode indicar uma mudança no padrão ou uma anomalia devido a poucos dados (como observado noano de 2022, que só tem dados do quarto trimestre).
- O sinistro que mais caiu em relação ao ano anterior foi o de Morte Acidental (2024: 5839 -> 2025: 4942)
---

In [0]:
window_ano_status = Window.partitionBy("ano", "status_seguro")

df_razao_status = df_tempo \
    .groupBy("ano", "TRIMESTRE", "status_seguro").count().withColumnRenamed("count", "qtd_trimestre") \
    .withColumn("total_ano", F.sum("qtd_trimestre").over(window_ano_status)) \
    .withColumn("razao_trimestral", F.col("qtd_trimestre") / F.col("total_ano")) \
    .orderBy("ano", "status_seguro", "trimestre")

display(df_razao_status)

### Resultado

De maneira geral, os tipos status se mantêm constantes ao longo dos trimestres e dos anos.

**2025**
- O mesmo padrão de poucos casos no quarto trimestre se repete.
